In [2]:
import polars as pl

df = pl.DataFrame({'x': [10], 'y': [2]})

expr_lit_2 = pl.lit(2)
expr_lit_2_int8 = pl.lit(2, dtype=pl.Int8)
expr_lit_2_int16 = pl.lit(2, dtype=pl.Int16)
expr_col_10 = pl.col('x')

expr_lit_10 = pl.lit(10)
expr_lit_10_int8 = pl.lit(10)
expr_lit_10_int16 = pl.lit(10)
expr_col_2 = pl.col('y')

## Failing cases
# Case 1: With Int8 dtype - FAILS (returns 0)
result1 = df.select(expr_lit_2).pow(expr_col_10))
result2 = df.select(expr_lit_2_int8).pow(expr_col_10))
result3 = df.select(expr_lit_2_int16).pow(expr_col_10))

result4 = df.select(expr_col_2).pow(expr_lit_10))
result5 = df.select(expr_col_2).pow(expr_lit_10_int8))
result6 = df.select(expr_col_2).pow(expr_lit_10_int16))


print("""result1: {result1}
result2: {result2}
result3: {result3}
result4: {result4}
result5: {result5}
result6: {result6}
""")

df2 = pl.DataFrame({'x': [2]})


# print(f"pl.lit(2, dtype=Int8).pow(col(10)) = {result1['literal'][0]}")
# Expected: 1024 or OverflowError
# Actual: 0

# Case 4: Using ** operator with Int8 - ALSO FAILS
result4 = df.select((pl.lit(2, dtype=pl.Int8) ** pl.col('x')))
print(f"pl.lit(2, dtype=Int8) ** col(10) = {result4['literal'][0]}")
# Expected: 1024
# Actual: 0 ❌

## Working cases

# Case 2: Without dtype - WORKS (returns 1024)
result2 = df.select(pl.lit(2).pow(pl.col('x')))
print(f"pl.lit(2).pow(col(10)) = {result2['literal'][0]}")
# Expected: 1024
# Actual: 1024 ✅

# Case 3: With Int16 dtype - WORKS (returns 1024)
result3 = df.select(pl.lit(2, dtype=pl.Int16).pow(pl.col('x')))
print(f"pl.lit(2, dtype=Int16).pow(col(10)) = {result3['literal'][0]}")
# Expected: 1024
# Actual: 1024 ✅


# Case 5: Using ** operator WITHOUT dtype - WORKS
result5 = df.select((pl.lit(2) ** pl.col('x')))
print(f"pl.lit(2) ** col(10) = {result5['literal'][0]}")
# Expected: 1024
# Actual: 1024 ✅

pl.lit(2, dtype=Int8).pow(col(10)) = 0
pl.lit(2, dtype=Int8) ** col(10) = 0
pl.lit(2).pow(col(10)) = 1024
pl.lit(2, dtype=Int16).pow(col(10)) = 1024
pl.lit(2) ** col(10) = 1024


In [3]:
result1 = df.select(pl.col('x').pow(pl.lit(2, dtype=pl.Int8)))
print(f"pl.lit(2, dtype=Int8).pow(col(10)) = {result1['literal'][0]}")
# Expected: 1024 or OverflowError
# Actual: 0



ColumnNotFoundError: "literal" not found

In [7]:
import polars as pl

def power_test(base: int|float, exp: int|float) -> pl.DataFrame:
    
    df = pl.DataFrame({'base': [base], 'exp': [exp]})
    
    #Literal Base
    lit_base = pl.lit(base)
    lit_base_int8 = pl.lit(base, dtype=pl.Int8)
    lit_base_int16 = pl.lit(base, dtype=pl.Int16)
    lit_base_float32 = pl.lit(base, dtype=pl.Float32)
    
    #Column Base
    col_base = pl.col('base')
    col_base_int8 = pl.col('base').cast(pl.Int8)
    col_base_int16 = pl.col('base').cast(pl.Int16)
    # col_base_float8 = pl.col('base').cast(pl.Float8)
    col_base_float32 = pl.col('base').cast(pl.Float32)
    
    #Literal Exponent
    lit_exp = pl.lit(exp)
    lit_exp_int8 = pl.lit(exp, dtype=pl.Int8)
    lit_exp_int16 = pl.lit(exp, dtype=pl.Int16)
    # lit_exp_float8 = pl.lit(exp, dtype=pl.Float8)
    lit_exp_float32 = pl.lit(exp, dtype=pl.Float32)
    
    #Column Exponent
    col_exp = pl.col('exp')
    col_exp_int8 = pl.col('exp').cast(pl.Int8)
    col_exp_int16 = pl.col('exp').cast(pl.Int16)
    # col_exp_float16 = pl.col('exp').cast(pl.Float8)
    col_exp_float32 = pl.col('exp').cast(pl.Float32)
    
    
    result = df.select(
        lit_base.alias("base"),
        lit_exp.alias("exponent"),
        # Cases with LHS Int8 (column or literal) - FAIL silently (returns 0) due to overflow
        # LHS: literal
        lit_base_int8.pow(col_exp).alias("fail1"),
        lit_base_int8.pow(col_exp_int8).alias("fail2"),
        lit_base_int8.pow(col_exp_int16).alias("fail3"),
        # LHS: column
        col_base_int8.pow(lit_exp).alias("fail4"),
        col_base_int8.pow(lit_exp_int8).alias("fail5"),
        col_base_int8.pow(lit_exp_int16).alias("fail6"),
        
        # Cases with LHS > Int8 - CORRECT
        # LHS: literal
        lit_base.pow(col_exp).alias("result1"),
        lit_base.pow(col_exp_int8).alias("result2"),
        lit_base.pow(col_exp_int16).alias("result3"),
        # LHS: column
        col_base.pow(lit_exp).alias("result4"),
        col_base.pow(lit_exp_int8).alias("result5"),
        col_base.pow(lit_exp_int16).alias("result6")
    
    )
    
    return result

print( power_test(2,8) )

shape: (1, 14)
┌──────┬──────────┬───────┬───────┬───┬────────────┬────────────┬────────────┬────────────┐
│ base ┆ exponent ┆ fail1 ┆ fail2 ┆ … ┆ result3    ┆ result4    ┆ result5    ┆ result6    │
│ ---  ┆ ---      ┆ ---   ┆ ---   ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ f64  ┆ i32      ┆ i8    ┆ i8    ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞══════╪══════════╪═══════╪═══════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 2.2  ┆ 8        ┆ 0     ┆ 0     ┆ … ┆ 548.758735 ┆ 548.758735 ┆ 548.758735 ┆ 548.758735 │
└──────┴──────────┴───────┴───────┴───┴────────────┴────────────┴────────────┴────────────┘


In [ ]:
## Failing cases
# Cases with LHS Int8 (column or literal) - FAIL silently (returns 0) due to overflow
fail1 = df.select(lit_2_int8.pow(col_10).alias("result"))
fail2 = df.select(lit_2_int8.pow(col_10_int8).alias("result"))
fail3 = df.select(lit_2_int8.pow(col_10_int16).alias("result"))

fail4 = df.select(col_2_int8.pow(lit_10).alias("result"))
fail5 = df.select(col_2_int8.pow(lit_10_int8).alias("result"))
fail6 = df.select(col_2_int8.pow(lit_10_int16).alias("result"))


result1 = df.select(lit_2.pow(col_10).alias("result"))
result2 = df.select(lit_2.pow(col_10_int8).alias("result"))
result3 = df.select(lit_2.pow(col_10_int16).alias("result"))
result4 = df.select(col_2.pow(lit_10).alias("result"))
result5 = df.select(col_2.pow(lit_10_int8).alias("result"))
result6 = df.select(col_2.pow(lit_10_int16).alias("result"))


print(f"""fail1: {fail1["result"]}
# fail2: {fail2.result}
# fail3: {fail3.result}
# fail4: {fail4.result}
# fail5: {fail5.result}
# fail6: {fail6.result}

# result1: {result1.result}
# result2: {result2.result}
# result3: {result3.result}
# result4: {result4.result}
# result5: {result5.result}
# result6: {result6.result}
""")

In [8]:
import polars as pl

df = pl.DataFrame({'base': [2], 'exp': [8]})

# Int8 base - silently returns 0 (should be 256)
result_int8 = df.select(
    pl.lit(2, dtype=pl.Int8).pow(pl.col('exp')).alias("int8_pow")
)
print(result_int8)  # Shows 0, not 256

# i32 base - correctly returns 256
result_i32 = df.select(
    pl.lit(2, dtype=pl.Int32).pow(pl.col('exp')).alias("i32_pow")
)
print(result_i32)  # Shows 256 ✓

shape: (1, 1)
┌──────────┐
│ int8_pow │
│ ---      │
│ i8       │
╞══════════╡
│ 0        │
└──────────┘
shape: (1, 1)
┌─────────┐
│ i32_pow │
│ ---     │
│ i32     │
╞═════════╡
│ 256     │
└─────────┘


In [4]:
import polars as pl

df = pl.DataFrame({'base': [2], 'exp': [8]})

# Expression definitions
lit_base_i8 = pl.lit(2, dtype=pl.Int8)
col_exp = pl.col('exp')

# Execution: Int8 base - silently returns 0 (should be 256)
result_int8 = df.select(
    lit_base_i8.pow(col_exp).alias("int8_pow")
)
print(result_int8)  # Shows 0, not 256 


shape: (1, 1)
┌──────────┐
│ int8_pow │
│ ---      │
│ i8       │
╞══════════╡
│ 0        │
└──────────┘


In [2]:
import polars as pl

df = pl.DataFrame({'base': [10], 'exp': [10]})

# Expression definitions
lit_base_i16 = pl.lit(10, dtype=pl.Int16)
lit_base_i64 = pl.lit(10, dtype=pl.Int64)
lit_exp_i16 = pl.lit(10, dtype=pl.Int16)

# Execution: Int16 wraps to -7168 (10^10 = 10,000,000,000 requires 34 bits)
result_int16 = df.select(
    lit_base_i16.pow(lit_exp_i16).alias("int16_pow")
)
print(result_int16)  # Shows -7168, not 10000000000 

# Execution: i64 base - correctly returns 10,000,000,000
result_i64 = df.select(
    lit_base_i64.pow(lit_exp_i16).alias("i64_pow")
)
print(result_i64)  # Shows 10000000000 ✓

shape: (1, 1)
┌───────────┐
│ int16_pow │
│ ---       │
│ i16       │
╞═══════════╡
│ -7168     │
└───────────┘
shape: (1, 1)
┌─────────────┐
│ i64_pow     │
│ ---         │
│ i64         │
╞═════════════╡
│ 10000000000 │
└─────────────┘


In [3]:
pl.show_versions()

--------Version info---------
Polars:              1.35.1
Index type:          UInt32
Platform:            Linux-6.17.0-6-generic-x86_64-with-glibc2.42
Python:              3.12.9 (main, Feb 12 2025, 14:50:50) [Clang 19.1.6 ]
Runtime:             rt32

----Optional dependencies----
Azure CLI            <not installed>
adbc_driver_manager  <not installed>
altair               5.5.0
azure.identity       <not installed>
boto3                1.34.113
cloudpickle          3.1.1
connectorx           <not installed>
deltalake            <not installed>
fastexcel            <not installed>
fsspec               2025.10.0
gevent               <not installed>
google.auth          <not installed>
great_tables         <not installed>
matplotlib           3.10.7
numpy                1.26.0
openpyxl             <not installed>
pandas               2.2.2
polars_cloud         <not installed>
pyarrow              17.0.0
pydantic             2.9.2
pyiceberg            <not installed>
sqlalchemy          